In [1]:
import re

import pandas as pd
import string

directory_path = 'STAR_eval/Charades'
from gpt_ask import run_gpt

In [2]:
main_ds = pd.read_json('STAR_eval/STAR_val.json')
main = main_ds.loc[:, ['question_id','question','video_id','start','end','answer', 'choices']]

In [3]:
df = pd.read_csv('STAR_eval/all_cured.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna()
df = df.drop_duplicates()

In [4]:
df['OIC_answer_without_CUID'] =  df['OIC_context'].apply(lambda x: re.sub('_[a-zA-Z0-9]*', '', x))
df['OIC_answer_without_CUID'][0]

'The scene opens with man wearing shirt\nman has hair\nman wearing pant\nman has arm\nman wearing glove\nman has head\nman has ear\nbottle on counter\nbottle in hand\nbottle on counter\nperson wearing glove\nhead of man\nman wearing hat\nwindow in room\nAfter 1 seconds:\nman has hand\nman wearing glove\nAfter 2 seconds:\nman has glove\nAfter 3 seconds:\nman wearing pant\nman wearing glove\nglove on hand\ncurtain on window\nAfter 4 seconds:\nperson wearing glove\nAfter 5 seconds:\nman has glove\nperson wearing jacket\nAfter 6 seconds:\nAfter 7 seconds:\nhair on man\nhand of man\nwindow in room\nAfter 8 seconds:\nman holding laptop\nbottle on counter\nbag on hand\nAfter 9 seconds:\nman wearing glove\nman using laptop\nman wearing glass\nbottle in hand\nAfter 10 seconds:\nhand of man\ncurtain on window\nAfter 11 seconds:\nman wearing glove\nshirt on man\nhair on man\nAfter 12 seconds:\nman wearing hat\near of man\nAfter 13 seconds:\nman wearing pant\nman wearing glove\nshirt on man\nperso

In [5]:
class Substitutable(str):
  def __new__(cls, *args, **kwargs):
    newobj = str.__new__(cls, *args, **kwargs)
    newobj.sub = lambda fro,to: Substitutable(re.sub(fro, to, newobj))
    return newobj

In [6]:
df['OIC_answer_without_CUID_temp'] = df['OIC_answer_without_CUID']
df['OIC_answer_without_CUID_temp'] = df['OIC_answer_without_CUID_temp'].apply(lambda x: Substitutable(x).sub('\n', ' ').sub('The scene opens with', ' ').sub('From now on', ' ').sub('is not longer actual.',' ').sub('\nAfter [0-9]* seconds:', ' '))

df['OIC_answer_without_CUID_temp']

0        man wearing shirt man has hair man wearing p...
1        woman has glass woman wearing shirt woman ha...
2        glass on face glass on woman woman wearing g...
3        man wearing shirt man wearing hat man wearin...
4        man in shirt man has hand man wearing pant m...
                             ...                        
951      man wearing jacket man has hair man holding ...
952      towel on counter man standing on room After ...
953      girl wearing shirt woman has hair woman wear...
954      man wearing shirt man has hair man wearing p...
955      man wearing shirt man in room man has hair m...
Name: OIC_answer_without_CUID_temp, Length: 956, dtype: object

In [7]:
df.head()

,question_id,question,video_id,start,end,answer,choices,OIC_context,OIC_answer,OIC_question,Match,OIC_answer_without_CUID,OIC_answer_without_CUID_temp
0,Interaction_T3_2962,What did the person do while they were holding...,P4HXN,26.8,33.3,Put down the laptop.,"[{'choice_id': 0, 'choice': 'Took the laptop.'...",The scene opens with man_821f wearing shirt_65...,2.0,What did the person do while they were holding...,Correct,The scene opens with man wearing shirt\nman ha...,man wearing shirt man has hair man wearing p...
1,Interaction_T3_2963,What did the person do while they were touchin...,UF91R,7.3,12.1,Closed the laptop.,"[{'choice_id': 0, 'choice': 'Closed the laptop...",The scene opens with woman_f924 has glass_8d30...,1.0,What did the person do while they were touchin...,Correct,The scene opens with woman has glass\nwoman we...,woman has glass woman wearing shirt woman ha...
2,Interaction_T3_2971,What did the person do while they were holding...,PQYWB,6.1,10.7,Put down the towel.,"[{'choice_id': 0, 'choice': 'Put down the towe...",The scene opens with glass_dab7 on face_7158\n...,1.0,What did the person do while they were holding...,Correct,The scene opens with glass on face\nglass on w...,glass on face glass on woman woman wearing g...
3,Interaction_T3_3111,What did the person do while they were holding...,N7GBK,2.0,6.0,Put down the towel.,"[{'choice_id': 0, 'choice': 'Put down the towe...",The scene opens with man_87a7 wearing shirt_22...,1.0,What did the person do while they were holding...,Correct,The scene opens with man wearing shirt\nman we...,man wearing shirt man wearing hat man wearin...
4,Interaction_T3_3120,What did the person do while they were holding...,8MLCU,4.4,10.2,Put down the box.,"[{'choice_id': 0, 'choice': 'Took the box.', '...",The scene opens with man_609c in shirt_b141\nm...,4.0,What did the person do while they were holding...,Correct,The scene opens with man in shirt\nman has han...,man in shirt man has hand man wearing pant m...


In [8]:
from time import sleep
import time
import numpy as np
q_ids = np.unique(df['question_id'])
print(len(q_ids))
for q in q_ids:
  que = df.query("question_id == '"+q+"'")
  question = que['question'].values[0]
  answer = que['answer'].values[0]
  choices = dict()
  choice_string = ''
  options = main.query("question_id == '"+q+"'")["choices"].values[0]
  for choice in options:
        choices.update({choice['choice_id']:choice['choice'].lower().strip('the').translate(str.maketrans('', '', string.punctuation)).strip()})
        choice_string += ' ('+str(choice['choice_id']+1)+')'+str(choice['choice'].lower().strip('the').translate(str.maketrans('', '', string.punctuation)))
  
  que_df = df.query("question_id == '"+q+"'")
  prompt = que_df['OIC_answer_without_CUID_temp'].values[0] # update column name to OIC_answer_without_CUID_temp
  formatted_question = question+' Guess the most likely answer among these four options: '+choice_string+' Respond only with a single number between 1 and 4. If not found in the given options reply 0 not any other statements.'
  response = run_gpt(prompt, formatted_question)
  while True:
    if len(response)>1:
        response = run_gpt(prompt, formatted_question) 
    else:
      response = int(response)
      break
  if int(response) > 0:
    OIC_answer = response
    if choices[int(OIC_answer)-1] in answer:
      df.loc[df['question_id'] == q, 'OIC_answer_without_CUID_temp'] = 'Correct' # update column name to Wo_CUID_temp
      print(df.query("question_id == '"+q+"'")['OIC_answer_without_CUID_temp'].values[0])
    else:
      df.loc[df['question_id'] == q, 'OIC_answer_without_CUID_temp'] = 'Wrong' # update column name to Wo_CUID_temp
      print(df.query("question_id == '"+q+"'")['OIC_answer_without_CUID_temp'].values[0])
  else:
    df.loc[df['question_id'] == q, 'OIC_answer_without_CUID_temp'] = 'Wrong'
  df.to_csv('STAR_eval/new_all_cured_2.csv') # update column name to without_cuid_temp  
  #df.drop(df[df['Wo_CUID_Match'].isnull().values.any()].index, inplace = True)
  #sleep(100)

909
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Correct
Correct
Correct
Correct
Wrong
Correct
Correct
Wrong
Wrong
Wrong
Wrong
Correct
Correct
Wrong
Correct
Correct
Wrong
Wrong
Wrong
Wrong
Correct
Wrong
Correct
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Correct
Wrong
Correct
Correct
Correct
Correct
Wrong
Correct
Correct
Correct
Correct
Wrong
Wrong
Correct
Wrong
Correct
Wrong
Correct
Wrong
Correct
Correct
Correct
Correct
Correct
Wrong
Wrong
Wrong
Correct
Wrong
Correct
Correct
Correct
Correct
Correct
Correct
Wrong
Wrong
Correct
Correct
Correct
Wrong
Wrong
Correct
Correct
Wrong
Correct
Correct
Wrong
Correct
Wrong
Wrong
Wrong
Correct
Wrong
Wrong
Wrong
Wrong
Correct
Wrong
Correct
Wrong
Correct
Wrong
Wrong
Wrong
Wrong
Correct
Wrong
Correct
Correct
Correct
Correct
Correct
Correct
Correct
Correct
Wrong
Wrong
Wrong
Wrong
Correct
Wrong
Wrong
Correct
Wrong
Correct
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Wrong
Correct
Wron